In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch , torch.nn as nn
import tensorflow as tf
from tensorflow import keras

In [16]:
from torch.utils.data import Dataset,DataLoader
df = pd.read_csv('/kaggle/input/competitions/histopathologic-cancer-detection/train_labels.csv')
img_dir = '/kaggle/input/competitions/histopathologic-cancer-detection/train/'


filepath = (img_dir + df['id']+'.tif').values
labels = df['label'].values
print(filepath)
print(labels)

def loadImage(filepath, label):
    img = tf.io.read_file(filepath)
    img = tfio.experimental.image.decode_tiff(img)
    img = img[:, :, :3]  
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img, label

dataset = tf.data.Dataset.from_tensor_slices((filepath, labels))
dataset = dataset.map(loadImage, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

['/kaggle/input/competitions/histopathologic-cancer-detection/train/f38a6374c348f90b587e046aac6079959adf3835.tif'
 '/kaggle/input/competitions/histopathologic-cancer-detection/train/c18f2d887b7ae4f6742ee445113fa1aef383ed77.tif'
 '/kaggle/input/competitions/histopathologic-cancer-detection/train/755db6279dae599ebb4d39a9123cce439965282d.tif'
 ...
 '/kaggle/input/competitions/histopathologic-cancer-detection/train/3d046cead1a2a5cbe00b2b4847cfb7ba7cf5fe75.tif'
 '/kaggle/input/competitions/histopathologic-cancer-detection/train/f129691c13433f66e1e0671ff1fe80944816f5a2.tif'
 '/kaggle/input/competitions/histopathologic-cancer-detection/train/a81f84895ddcd522302ddf34be02eb1b3e5af1cb.tif']
[0 1 0 ... 0 0 1]


NotImplementedError: in user code:

    File "/tmp/ipykernel_58/504860804.py", line 13, in loadImage  *
        img = tfio.experimental.image.decode_tiff(img)
    File "/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/experimental/image_ops.py", line 87, in decode_tiff  **
        return core_ops.io_decode_tiff(contents, index, name=name)
    File "/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py", line 88, in __getattr__
        return getattr(self._load(), attrb)
    File "/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py", line 84, in _load
        self._mod = _load_library(self._library)
    File "/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py", line 69, in _load_library
        raise NotImplementedError(

    NotImplementedError: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
    caused by: 

In [17]:
import tensorflow as tf
import cv2
import numpy as np

def _load_image_numpy(filepath):
    filepath = filepath.numpy().decode('utf-8')
    img = cv2.imread(filepath)  # BGR, uint8
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    return img

def loadImage(filepath, label):
    img = tf.py_function(func=_load_image_numpy, inp=[filepath], Tout=tf.float32)
    img.set_shape([96, 96, 3]) 
    label.set_shape([])
    return img, label

dataset = tf.data.Dataset.from_tensor_slices((filepath, labels))
dataset = dataset.map(loadImage, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

In [18]:
for img, label in dataset.take(1):
    print(img.shape, label.numpy())

(32, 96, 96, 3) [1 0 1 1 0 0 1 1 0 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 1 1 0 0 0 1 1 0]


In [19]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    filepath, labels, test_size=0.2, stratify=labels, random_state=42
)

train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.map(loadImage, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(2000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(loadImage, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

In [20]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(96,96,3))
base_model.trainable = False  

inputs = layers.Input(shape=(96,96,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [21]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

Epoch 1/10


2026-09-09 15:19:22.013726: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:19:22.147147: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:19:22.463556: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:19:22.604348: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:19:23.446894: E external/local_xla/xla/stream_

5498/5501 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - auc: 0.5069 - loss: 0.6768

2026-09-09 15:27:34.924163: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:27:35.058104: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:27:35.374635: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:27:35.515903: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:27:36.336495: E external/local_xla/xla/stream_

5501/5501 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - auc: 0.5069 - loss: 0.6768

2026-09-09 15:29:48.449222: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:29:48.582259: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:29:48.882563: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:29:49.024158: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 15:29:49.779542: E external/local_xla/xla/stream_

5501/5501 ━━━━━━━━━━━━━━━━━━━━ 657s 113ms/step - auc: 0.5102 - loss: 0.6754 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 2/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 282s 51ms/step - auc: 0.5035 - loss: 0.6748 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 3/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 291s 52ms/step - auc: 0.4982 - loss: 0.6750 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 4/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 287s 52ms/step - auc: 0.4983 - loss: 0.6750 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 5/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 287s 52ms/step - auc: 0.4983 - loss: 0.6750 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 6/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 284s 51ms/step - auc: 0.4999 - loss: 0.6750 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 7/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 285s 51ms/step - auc: 0.4983 - loss: 0.6750 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 8/10
5501/5501 ━━━━━━━━━━━━━━━━━━━━ 280s 51ms/step - auc: 0.4983 - loss: 0.6750 - val_auc: 0.5000 - val_loss: 0.6750
Epoch 9/10
5501/5501 ━━━━━